## Generating the Claw Coupler

In [1]:
#Enables module automatic reload. 
#Your notebook will be able to pick up code updates made to qiskit-metal (or other) module code.

%reload_ext autoreload
%autoreload 2

In [2]:
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, Headings

In [15]:
import numpy as np
from qiskit_metal.qlibrary.core import QComponent

class ClawCoupler(QComponent):
    """A standalone claw-style CPW coupler."""
    
    default_options = Dict(
        claw_length='30um',
        claw_spacing='5um',
        claw_width='10um',
        claw_gap='6um',
        claw_cpw_length='40um',
        claw_cpw_width='10um',
        claw_tip_width='1um',
        claw_tip_spacing='1um'
    )
    """Default options."""
    
    component_metadata = Dict(short_name='ClawCoupler')
    """Component metadata."""
    
    TOOLTIP = """A standalone claw-style coupler for CPW transmission lines."""
    
    def make(self):
        """Generates the geometry for the claw coupler."""
        p = self.p
        
        c_g = p.claw_gap
        c_l = p.claw_length
        c_w = p.claw_width
        c_c_w = p.claw_cpw_width
        c_c_l = p.claw_cpw_length
        c_s = p.claw_spacing
        c_t_w = p.claw_tip_width
        c_t_s = p.claw_tip_spacing
        
        t_claw_height = 2 * c_w + c_s
        
        claw_cpw = draw.box(-c_w, -c_c_w / 2, -c_c_l - c_w, c_c_w / 2)
        claw_base = draw.box(-c_w, -(t_claw_height) / 2, c_l, t_claw_height / 2)
        claw_subtract = draw.box(0, -t_claw_height / 2 + c_w, c_l - c_t_w, t_claw_height / 2 - c_w)
        claw_base = claw_base.difference(claw_subtract)
        claw_tip_subtract = draw.box(c_l - c_t_w, -c_t_s / 2, c_l, c_t_s / 2)
        claw_base = claw_base.difference(claw_tip_subtract)
        port_line = draw.LineString([(-c_c_l - c_w, -c_c_w / 2), (-c_c_l - c_w, c_c_w / 2)])
        
        connector_arm = draw.shapely.ops.unary_union([claw_base, claw_cpw])
        connector_etcher = draw.buffer(connector_arm, c_g)
        connector_etcher = draw.shapely.ops.unary_union([connector_etcher, claw_subtract, claw_tip_subtract])
        
        polys = [connector_arm, connector_etcher, port_line]
        polys = draw.rotate(polys, p.orientation, origin=(0, 0))
        polys = draw.translate(polys, p.pos_x, p.pos_y)
        [connector_arm, connector_etcher, port_line] = polys
        
        # Generate qgeometry for the claw coupler
        self.add_qgeometry('poly', {'claw_connector_arm': connector_arm}, chip=p.chip)
        self.add_qgeometry('poly', {'claw_connector_etcher': connector_etcher}, subtract=True, chip=p.chip)
        
        # Define pin for connections
        self.add_pin('claw', port_line.coords, c_c_w)

## Generating a launch pad + CPW, and connect to the Claw Coupler

In [16]:
design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = '3mm'
design.chips.main.size['size_y'] = '3mm'

gui = MetalGUI(design)

# If you disable the next line with "overwrite_enabled", then you will need to 
# delete a component [<component>.delete()] before recreating it.
design.overwrite_enabled = True

In [17]:
from qiskit_metal.qlibrary.terminations.launchpad_wb_coupled import LaunchpadWirebondCoupled

#Explore the options of the LaunchpadWirebondCoupled.
LaunchpadWirebondCoupled.get_template_options(design)

from qiskit_metal.qlibrary.tlines.meandered import RouteMeander

#Explore the options of the RouteMeander.
RouteMeander.get_template_options(design)

{'chip': 'main',
 'layer': '1',
 'pin_inputs': {'start_pin': {'component': '', 'pin': ''},
  'end_pin': {'component': '', 'pin': ''}},
 'fillet': '0',
 'lead': {'start_straight': '0mm',
  'end_straight': '0mm',
  'start_jogged_extension': '',
  'end_jogged_extension': ''},
 'total_length': '7mm',
 'trace_width': 'cpw_width',
 'meander': {'spacing': '200um', 'asymmetry': '0um'},
 'snap': 'true',
 'prevent_short_edges': 'true',
 'hfss_wire_bonds': False,
 'q3d_wire_bonds': False}

In [18]:
lp = LaunchpadWirebondCoupled(design, 'launch1', options=dict(pos_x='-1.5mm', pos_y='0mm'))
Q1 = ClawCoupler(design, 'claw1', 
                 options=dict(pos_x='0.5mm', pos_y='0mm',
                              claw_spacing = '40um',claw_width='5um',
                             claw_tip_width='0um',claw_tip_spacing='0.5um'))
# After adding the components, create the meander
meander_options = Dict(
    total_length='10mm',
    fillet='90um',
    lead=Dict(start_straight='100um', end_straight='100um'),
    pin_inputs=Dict(
        end_pin=Dict(component=lp.name, pin='tie'),
        start_pin=Dict(component=Q1.name, pin='claw')
    )
)

meander = RouteMeander(design, 'bus', options=meander_options)

gui.rebuild()
gui.autoscale()

In [ ]:
# Get a list of all the qcomponents in QDesign and then zoom on them.
all_component_names = design.components.keys()

gui.zoom_on_components(all_component_names)

In [ ]:
all_component_names